In [1]:
import re
import requests
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

# Configuration de spaCy
nlp = spacy.load("en_core_web_sm")

urls = [
    "https://www.gutenberg.org/files/11/11-0.txt",       
    "https://www.gutenberg.org/files/12/12-0.txt",       
    "https://www.gutenberg.org/cache/epub/29042/pg29042.txt" 
]
titles = ["Alice in Wonderland", "Through the Looking-Glass", "A Tangled Tale"]


# 1 & 2. Chargement et nettoyage (Slicing pour enlever les métadonnées Gutenberg)
def load_texts(urls_list):
    corpus = []
    for url in urls_list:
        response = requests.get(url)
        # Gestion de l'encodage UTF-8 (parfois avec BOM)
        response.encoding = 'utf-8-sig' 
        text = response.text
        
        # Slicing pour supprimer les crédits du Projet Gutenberg en début et fin
        start_match = re.search(r"\*\*\* START OF TH(E|IS) PROJECT GUTENBERG", text)
        end_match = re.search(r"\*\*\* END OF TH(E|IS) PROJECT GUTENBERG", text)
        
        if start_match and end_match:
            text = text[start_match.end():end_match.start()]
            
        # Nettoyage des caractères non-alphabétiques (en gardant les espaces)
        cleaned_text = re.sub(r'[^a-zA-Z\s]', '', text)
        # Remplacement des sauts de ligne multiples par un espace unique
        cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
        
        corpus.append(cleaned_text)
    return corpus

print("--- Étape 1 & 2 : Chargement des livres ---")
raw_corpus = load_texts(urls)

for title, text in zip(titles, raw_corpus):
    print(f"\n> Premier 200 caractères de '{title}':\n{text[:200]}...")

# 3. Tokenization
print("\n--- Étape 3 : Tokenization ---")
tokenized_corpus = [word_tokenize(text.lower()) for text in raw_corpus]
for title, tokens in zip(titles, tokenized_corpus):
    print(f"> 150 premiers tokens de '{title}':\n{tokens[:150]}\n")

# 4. Suppression des Stopwords
print("--- Étape 4 : Suppression des Stopwords ---")
stop_words = set(stopwords.words('english'))
filtered_corpus = [[w for w in tokens if w not in stop_words] for tokens in tokenized_corpus]

for title, tokens, filtered in zip(titles, tokenized_corpus, filtered_corpus):
    print(f"'{title}' -> Avant : {len(tokens)} tokens | Après : {len(filtered)} tokens")
    # Vérification d'un stopword précis
    print(f"Nombre d'occurrences du mot 'i' après nettoyage : {filtered.count('i')}")

# 5. Stemming (Porter Stemmer)
print("\n--- Étape 5 : Stemming (Porter) ---")
stemmer = PorterStemmer()
stemmed_corpus = [[stemmer.stem(w) for w in tokens] for tokens in filtered_corpus]
for title, stems in zip(titles, stemmed_corpus):
    print(f"> 50 premiers stems de '{title}':\n{stems[:50]}\n")

# 6. Lemmatisation (spaCy)
print("--- Étape 6 : Lemmatisation (spaCy) ---")
# Pour spaCy, on traite une portion du texte filtré pour aller plus vite
lemmatized_corpus = []
for text in raw_corpus:
    # On passe le texte brut (ou filtré reconstruit) à spaCy
    # Pour respecter la consigne des 50 premiers, on prend un échantillon du début
    doc = nlp(" ".join(word_tokenize(text.lower())[:200]))
    lemmas = [token.lemma_ for token in doc if token.text not in stop_words]
    lemmatized_corpus.append(lemmas)

for title, lemmas in zip(titles, lemmatized_corpus):
    print(f"> 50 premiers lemmes de '{title}':\n{lemmas[:50]}\n")

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.